# Step 2 — Tools

## What is a Tool?

A tool is a function that an agent can use to interact with the outside world.

Examples:

- Calculator
- Weather API
- Database query
- Search
- Python execution
- File access
- Kubernetes API

Conceptually:

Agent → Tool → Result

The LLM decides **when** a tool is needed.
The tool performs the actual operation.

Let's create the simplest possible tool:

In [3]:
def add(a, b):
    return a + b

In [4]:
result = add(10, 20)

print(result)

30


## Tool vs Agent

A tool:

- performs an operation
- does not decide what to do
- is usually deterministic

An agent:

- decides what action to take
- decides which tool to use
- interprets tool results
- may perform multiple actions

Example:

User:
"What is 25 * 17?"

Agent:
"I should use the calculator."

Tool:
25 * 17 = 425

Agent:
"The answer is 425."

In [5]:
# caluculator tool
def calculator(expression):
    return eval(expression)

In [6]:
calculator("25 * 17")

425

In [7]:
calculator("100 / 4")

25.0

In [8]:
calculator("(25 + 5) * 10")

300

# Agent Loop

An agent typically follows this cycle:

1. Receive task
2. Reason about what to do
3. Select an action/tool
4. Execute the tool
5. Observe the result
6. Decide whether more actions are required
7. Produce final answer

This creates a loop:
```
Task
 ↓
Think
 ↓
Act
 ↓
Observe
 ↓
Think
 ↓
Act
 ↓
Observe
 ↓
Final Answer

```

## Step 10 — Manually Implement the Loop

For now, we will pretend the LLM decided to call the calculator.

In [9]:
user_question = "What is 25 * 17?"

print("User:", user_question)

# Agent decides to use calculator
tool_input = "25 * 17"

# Execute tool
tool_result = calculator(tool_input)

print("Tool result:", tool_result)

User: What is 25 * 17?
Tool result: 425


## Step 11 — Give the Result Back to the LLM

In [11]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()
def ask_llm(prompt):
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )
    
    return response.output_text
# ========
# ========

prompt = f"""
The user asked:

{user_question}

A calculator tool was used.

The calculator returned:

{tool_result}

Give the user the final answer.
"""

answer = ask_llm(prompt)

print(answer)

The final answer is 425.


# Key realization

We still don't have a real agent.

We manually decided:
```
tool_input = "25 * 17"
```

The next step is to remove that hard-coded decision.

We will make the LLM itself decide whether to call the tool.

That introduces tool/function calling, which is the critical foundation for real agents.

# Step 3 — LLM Tool Calling

## Why Do We Need Tool Calling?

So far, we manually decided which tool to call:

```python
tool_input = "25 * 17"
tool_result = calculator(tool_input)

This is not really agentic because our Python code made the decision.

A real agent should allow the LLM to decide:

User
  ↓
LLM
  ↓
Should I use a tool?
  ↓
Yes
  ↓
Which tool?
  ↓
Calculator
  ↓
What arguments?
  ↓
25 * 17

## What Is Tool Calling?

Tool calling allows an LLM to return a structured request saying:

"I want the application to execute this function with these arguments."

For example, instead of generating:

The answer is 425.

the model can produce something conceptually like:

```
{
  "tool": "calculator",
  "arguments": {
    "expression": "25 * 17"
  }
}
```
The application then executes the function:

```
calculator("25 * 17")
```

and sends the result back to the LLM.

## The Tool-Calling Loop

The basic architecture becomes:


                    ┌──────────────┐
                    │     User     │
                    └──────┬───────┘
                           ↓
                    ┌──────────────┐
                    │     LLM      │
                    └──────┬───────┘
                           ↓
                    Decide what to do
                           ↓
                 ┌─────────┴─────────┐
                 │                   │
              No tool             Tool needed
                 │                   │
                 ↓                   ↓
            Final answer       Tool call request
                                     ↓
                              ┌──────────────┐
                              │    Tool      │
                              └──────┬───────┘
                                     ↓
                                  Result
                                     ↓
                              ┌──────────────┐
                              │     LLM      │
                              └──────┬───────┘
                                     ↓
                               Final answer

# Tool Calling vs Agent

Tool calling is a capability.

An agent is a system that uses capabilities such as tool calling to create an execution loop.

Therefore:
```
Tool Calling
     ↓
LLM can request actions
```

while:

```
Agent
 ↓
LLM
 ↓
Decide
 ↓
Tool
 ↓
Observe
 ↓
Decide again
 ↓
Tool
 ↓
Observe
 ↓
Final answer
```

The agent is therefore more than just a single tool call.

# The Core Agent Pattern

The fundamental pattern we are building is:
```
THINK → ACT → OBSERVE → THINK → ACT → OBSERVE → ANSWER
```
Where:
```
THINK = LLM decides what should happen
ACT = execute a tool
OBSERVE = receive the tool result
ANSWER = produce the final response
```
This loop is the foundation of many agentic systems.

# Step 4 — Building Our First Real Agent

Until now, we manually decided when to call the calculator.

Now we will let the LLM make that decision.

Our architecture will be:

```
User
 ↓
LLM
 ↓
Decide whether a tool is required
 ↓
Tool Call
 ↓
Python executes the tool
 ↓
Tool Result
 ↓
LLM
 ↓
Final Answer
```

The important difference is:

Before:
```
Python → decides → tool
```

Now:
```
LLM → decides → tool
```
Our Python application becomes the **agent runtime** that executes the actions requested by the LLM.

In [12]:
# Define the tool
def calculator(expression):
    return eval(expression)
calculator("25 * 17")

425

# 2. Describe the tool to the LLM

The LLM needs to know:

tool name
what the tool does
what arguments it accepts

In [13]:
tools = [
    {
        "type": "function",
        "name": "calculator",
        "description": "Evaluate a mathematical expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression such as '25 * 17'"
                }
            },
            "required": ["expression"],
            "additionalProperties": False
        },
        "strict": True
    }
]

This is the tool schema.

The Python function is:

calculator(expression)

The schema tells the LLM how it can request that function.

In [14]:
response = client.responses.create(
    model="gpt-5.6",
    input="What is 25 * 17?",
    tools=tools
)

print(response)

Response(id='resp_09ea7bcbff8c2a58006aa2b5ce35bc87d0a694d656165f27a4', created_at=1789048270.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-sol', object='response', output=[ResponseFunctionToolCall(arguments='{"expression":"25 * 17"}', call_id='call_pPnOhDMV0KjFddid7goiB3xz', name='calculator', type='function_call', id='fc_09ea7bcbff8c2a58006aa2b5d16e0c87d09043ea23af9fd87d', async_=None, caller=None, namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='calculator', parameters={'type': 'object', 'properties': {'expression': {'type': 'string', 'description': "A mathematical expression such as '25 * 17'"}}, 'required': ['expression'], 'additionalProperties': False}, strict=True, type='function', allowed_callers=None, async_=None, defer_loading=None, description='Evaluate a mathematical expression.', output_schema=None)], top_p=0.98, background=False, completed_at=1789048273.0,

At this point, don't assume the LLM directly executed calculator().

Instead, inspect what the model returned:

In [15]:
for item in response.output:
    print(item)

ResponseFunctionToolCall(arguments='{"expression":"25 * 17"}', call_id='call_pPnOhDMV0KjFddid7goiB3xz', name='calculator', type='function_call', id='fc_09ea7bcbff8c2a58006aa2b5d16e0c87d09043ea23af9fd87d', async_=None, caller=None, namespace=None, status='completed')


## Step 14 — Execute the Tool

Now Python needs to inspect the model's request and execute the corresponding function.

In [16]:
import json

for item in response.output:

    if item.type == "function_call":

        tool_name = item.name
        arguments = json.loads(item.arguments)

        print("Tool:", tool_name)
        print("Arguments:", arguments)

Tool: calculator
Arguments: {'expression': '25 * 17'}


In [17]:
tool_result = calculator(arguments["expression"])

print(tool_result)

425


## Step 15 — Send the Tool Result Back to the LLM

The LLM requested:

calculator("25 * 17")

We executed it and got:

425

Now we need to tell the LLM what happened.

```
response = client.responses.create(
    model="gpt-5.6",
    input=[
        {
            "role": "user",
            "content": "What is 25 * 17?"
        },
        {
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": str(tool_result)
        }
    ],
    tools=tools
)

print(response.output_text)
```

In [20]:
def run_agent(user_input):

    input_items = [
        {
            "role": "user",
            "content": user_input
        }
    ]

    while True:

        response = client.responses.create(
            model="gpt-5.6",
            input=input_items,
            tools=tools
        )

        # Add the model's response to the conversation
        input_items += response.output

        # Check whether the model requested a tool
        tool_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        # No tool call means the agent is finished
        if not tool_calls:
            return response.output_text

        # Execute requested tools
        for tool_call in tool_calls:

            if tool_call.name == "calculator":

                arguments = json.loads(tool_call.arguments)

                result = calculator(
                    arguments["expression"]
                )

                input_items.append(
                    {
                        "type": "function_call_output",
                        "call_id": tool_call.call_id,
                        "output": str(result)
                    }
                )

In [22]:
answer = run_agent(
    "What is 125 * 37?"
)

print(answer)

4,625


In [23]:
print(run_agent("What is (125 + 75) * 4?"))

800


# What Makes This an Agent?

Our code now has the essential agentic loop:

              ┌──────────────┐
              │     User     │
              └──────┬───────┘
                     ▼
              ┌──────────────┐
         ┌───►│     LLM      │
         │    └──────┬───────┘
         │           │
         │       Tool call?
         │           │
         │      ┌────┴────┐
         │      │         │
         │     YES        NO
         │      │         │
         │      ▼         ▼
         │    Tool      Answer
         │      │
         │      ▼
         │    Result
         │      │
         └──────┘



The key code is:
```
while True:
```
That loop allows the LLM to repeatedly:
```
decide
  ↓
act
  ↓
observe
  ↓
decide again
```
This is the fundamental Agent Loop.